# FFT basics, and the checks that the machinery is right

Before anything is measured on XAUUSD, known signals must come back where they were put. This notebook calls the library's positive controls; it computes nothing itself.

* **Frequency** is in cycles per bar, `f = k/N`; the period is `N/k` bars. The longest period an `N`-bar window can show is `N` bars, the shortest 2 bars (the Nyquist frequency, 1/2 cycle per bar).
* **Leakage**: a cycle that does not fit a whole number of times in the window spreads its power into neighbouring bins. The Hann taper reduces that; the price is a wider main lobe.
* **Aliasing**: bars are samples. A faster oscillation that bar aggregation did not average away appears at a lower, wrong frequency, so no sub-bar cycle is ever reported.
* **Raw price is not an input.** Its FFT is dominated by the trend (power piles up at the lowest bins) and says nothing about cycles; the study uses the regression residual, the OU innovation and log returns instead.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant.features.spectral_config import load_spectral_config
from xauusd_quant.research.fft_analysis import (
    aliased_frequency, positive_control_table, power_spectral_density, recover_components,
    resolution_table, sinusoid, window_spectrum)

spectral = load_spectral_config(ROOT / "config" / "spectral.yaml")

## Known sinusoids through the engine: injected vs recovered

On-bin frequencies come back exactly. An off-bin frequency (0.4 bin off the grid) is reported at the nearest bin, with some amplitude lost to scalloping and a phase biased by about pi x 0.4 - a real limitation of bin-grid estimates, shown rather than hidden.

In [ ]:
positive_control_table(spectral)

## Frequency resolution: when can two close cycles be told apart?

In [ ]:
resolution_table(spectral).pivot(on="fft_window", index="separation_cycles_per_bar",
                                  values="resolved")

## Aliasing: a 0.8 cycle-per-bar oscillation sampled once per bar

In [ ]:
print("appears at", aliased_frequency(0.8), "cycles per bar")
window_spectrum(sinusoid(200, [(0.8, 1.0, 0.0)]), spectral).sort("power", descending=True).head(3)

## Raw power, power share and power spectral density are different things

In [ ]:
x = sinusoid(512, [(16 / 512, 1.0, 0.0)], noise_std=0.5, seed=1)
display_rows = recover_components(x, spectral, 3)
psd = power_spectral_density(x, bar_seconds=300.0)
display_rows, psd.head()